# Crawling Detik.com

In [1]:
# Jika belum install library, jalankan sekali saja
!pip install requests beautifulsoup4 trafilatura pandas openpyxl tqdm

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [o

In [11]:
import re
import time
import random
import requests
import pandas as pd
import trafilatura

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlsplit, urlunsplit
from tqdm import tqdm


# ============================================================
# 1. KONFIGURASI
# ============================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(HEADERS)


kategori = {
    "sport": {
        "url": "https://sport.detik.com/indeks",
        "domain": "sport.detik.com"
    },

    "finance": {
        "url": "https://finance.detik.com/indeks",
        "domain": "finance.detik.com"
    }
}

In [12]:
def normalize_url(url):
    """
    Menghapus query parameter dan fragment dari URL.
    """

    bagian = urlsplit(url)

    url_bersih = urlunsplit(
        (
            bagian.scheme,
            bagian.netloc,
            bagian.path,
            "",
            ""
        )
    )

    return url_bersih

In [13]:
def ambil_link_berita(index_url, domain, target=150, max_page=20):

    links = []
    sudah_ada = set()

    for page in range(1, max_page + 1):

        url_page = f"{index_url}?page={page}"

        print(f"Membaca halaman {page}")

        try:

            response = session.get(
                url_page,
                timeout=20
            )

            response.raise_for_status()

        except Exception as e:

            print("Error:", e)
            continue


        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )


        for tag in soup.find_all("a", href=True):

            href = urljoin(
                url_page,
                tag["href"]
            )

            href = normalize_url(href)

            parsed = urlsplit(href)


            # URL artikel Detik biasanya memiliki /d-xxxx
            if (
                parsed.netloc == domain
                and re.search(r"/d-\d+", parsed.path)
            ):

                if href not in sudah_ada:

                    sudah_ada.add(href)

                    links.append(href)


        print(
            "Jumlah link:",
            len(links)
        )


        if len(links) >= target:
            break


        # delay
        time.sleep(
            random.uniform(1, 2)
        )


    return links

In [14]:
link_sport = ambil_link_berita(
    kategori["sport"]["url"],
    kategori["sport"]["domain"],
    target=150
)

print(
    "Total kandidat Sport:",
    len(link_sport)
)

Membaca halaman 1
Jumlah link: 20
Membaca halaman 2
Jumlah link: 40
Membaca halaman 3
Jumlah link: 59
Membaca halaman 4
Jumlah link: 78
Membaca halaman 5
Jumlah link: 97
Membaca halaman 6
Jumlah link: 117
Membaca halaman 7
Jumlah link: 137
Membaca halaman 8
Jumlah link: 156
Total kandidat Sport: 156


In [15]:
link_finance = ambil_link_berita(
    kategori["finance"]["url"],
    kategori["finance"]["domain"],
    target=150
)

print(
    "Total kandidat Finance:",
    len(link_finance)
)

Membaca halaman 1
Jumlah link: 17
Membaca halaman 2
Jumlah link: 37
Membaca halaman 3
Jumlah link: 57
Membaca halaman 4
Jumlah link: 77
Membaca halaman 5
Jumlah link: 96
Membaca halaman 6
Jumlah link: 114
Membaca halaman 7
Jumlah link: 134
Membaca halaman 8
Jumlah link: 150
Total kandidat Finance: 150


In [16]:
def bersihkan_teks(teks):

    if teks is None:
        return None


    # menghapus teks iklan
    teks = re.sub(
        r"ADVERTISEMENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    teks = re.sub(
        r"SCROLL TO CONTINUE WITH CONTENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    # menghapus spasi berlebihan
    teks = re.sub(
        r"\s+",
        " ",
        teks
    )


    return teks.strip()

In [17]:
def ambil_isi_berita(url):

    try:

        response = session.get(
            url,
            timeout=25
        )

        response.raise_for_status()


        isi = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=False,
            output_format="txt",
            favor_precision=True,
            deduplicate=True
        )


        isi = bersihkan_teks(isi)


        # menghindari halaman kosong
        # atau isi yang terlalu pendek
        if isi and len(isi) >= 300:

            return isi


    except Exception as e:

        print(
            "Gagal:",
            url
        )

        print(
            "Error:",
            e
        )


    return None

In [18]:
def scraping_kategori(
    links,
    label,
    target=100
):

    hasil = []


    for url in tqdm(
        links,
        desc=f"Scraping {label}"
    ):


        if len(hasil) >= target:
            break


        isi = ambil_isi_berita(url)


        if isi is not None:

            hasil.append(
                {
                    "isi_berita": isi,
                    "label": label
                }
            )


        # delay supaya tidak terlalu cepat
        time.sleep(
            random.uniform(1, 2)
        )


    print(
        f"{label} berhasil:",
        len(hasil)
    )


    return hasil

In [19]:
data_sport = scraping_kategori(
    link_sport,
    "sport",
    target=100
)


Scraping sport:  66%|██████▌   | 103/156 [04:12<02:09,  2.45s/it]

sport berhasil: 100


In [20]:
data_finance = scraping_kategori(
    link_finance,
    "finance",
    target=100
)


Scraping finance:  70%|███████   | 105/150 [03:44<01:36,  2.14s/it]

finance berhasil: 100


In [21]:
data_semua = (
    data_sport
    +
    data_finance
)

In [22]:
df = pd.DataFrame(
    data_semua
)

In [23]:
df.insert(
    0,
    "id",
    range(
        1,
        len(df) + 1
    )
)

In [24]:
df = df[
    [
        "id",
        "isi_berita",
        "label"
    ]
]

In [25]:
df

,id,isi_berita,label
0,1,Marco Bezzecchi tak sabar menghadapi MotoGP Sa...,sport
1,2,Marc Marquez masih harus beradaptasi dengan ko...,sport
2,3,Pebalap Mercedes GP Kimi Antonelli tampil luar...,sport
3,4,Selesai sudah Seri IV M-15 Men's World Tennis ...,sport
4,5,Jete Run Festival 2026 baru saja selesai digel...,sport
...,...,...,...
195,196,Layanan kereta api (KA) Bandara Soekarno-Hatta...,finance
196,197,Transmart Full Day Sale kembali hadir pada Min...,finance
197,198,Sebanyak enam bandara ditutup sementara imbas ...,finance
198,199,Menteri Perhubungan Dudy Purwagandhi menyampai...,finance


In [26]:
print(
    "Total data:",
    len(df)
)

Total data: 200


In [27]:
print(
    df["label"].value_counts()
)

label
sport      100
finance    100
Name: count, dtype: int64


In [28]:
nama_file = "dataset_detik_200_berita.xlsx"


df.to_excel(
    nama_file,
    index=False
)


print(
    "File berhasil disimpan:",
    nama_file
)

File berhasil disimpan: dataset_detik_200_berita.xlsx


In [29]:
import os

print(os.path.abspath("dataset_detik_200_berita.xlsx"))

C:\Users\FAHRIZAL UMAM\anaconda\dataset_detik_200_berita.xlsx
